# 03 · Uncertainty — does the error bar shrink with sample size?

Two different "how sure are we" questions, two different objects:

1. **v0's BART posterior interval** (Task A) — turns out to be ~**flat in PA**: it measures
   surface uncertainty at a contact profile, *not* sample size. Wrong object for "how good is
   this hitter."
2. **A bootstrap over each player's plate appearances** (`player_ci`) — correctly **narrows
   with PA**, like a real confidence interval.

Loads `results/task_a/` and `results/player_ci/`.

In [ ]:
# --- setup: find repo root, load helpers (run this first) ---
import sys, json
from pathlib import Path
import polars as pl
from IPython.display import Image, Markdown, display

pl.Config.set_tbl_rows(30)
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "config.yaml").exists()), here)
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

def jload(rel):
    return json.loads((RESULTS / rel).read_text())

def show_fig(rel, width=680):
    p = RESULTS / rel
    return Image(filename=str(p), width=width) if p.exists() else Markdown(f"_missing figure: {rel}_")

print("repo root:", ROOT)
print("results:  ", RESULTS, "(exists)" if RESULTS.exists() else "(MISSING)")

## Task A — v0's model interval does NOT shrink with PA

In [ ]:
ta = jload("task_a/task_a_metrics.json")
print("n player-seasons:", ta["n_player_seasons"], " median PA:", ta["median_pa"])
print("median interval width:", round(ta["median_width"], 4))
print("log(width) ~ log(PA) slope:", round(ta["width_vs_pa_loglog"]["slope"], 3),
      "  (ideal 1/sqrt(PA) would be -0.5; ~0 means flat)")
print("width correlates with PA:      r =", round(ta["width_r_pa"], 3))
print("width correlates with xwOBA:   r =", round(ta["width_r_xwoba_mean"], 3), " (tracks contact profile, not sample size)")
print("Savant inside the 90% CI:      ", round(ta["savant_in_ci90_fraction"], 3))

In [ ]:
show_fig("task_a/figures/interval_width_vs_pa.png")

Interval width vs PA is flat — the model band is the same width for a 100-PA player and a 600-PA player. That is *not* what you want from a 'how sure are we' band.

## player_ci — a bootstrap band that DOES shrink with PA

In [ ]:
ci = jload("player_ci/ci_metrics.json")
print("log(width) ~ log(PA) slope:", round(ci["loglog_slope_width_vs_pa"]["slope"], 3), " (near the ideal -0.5)")
print("bootstrap width vs analytic SE:  r =", round(ci["corr_bootstrap_width_vs_analytic_se"], 3), " (genuine sampling band)")
# bootstrap vs model interval width, by PA-bin midpoint
rows = [{"PA (bin midpoint)": db["pa_mid"],
         "bootstrap width": round(db["median"], 3),
         "v0 model width": round(dm["median"], 3)}
        for db, dm in zip(ci["bootstrap_width_bin_medians"], ci["model_width_bin_medians"])]
pl.DataFrame(rows)

In [ ]:
show_fig("player_ci/figures/width_vs_pa_bootstrap_vs_model.png")

The two bands **cross at ~400 PA**: below that, v0's flat band is up to ~1.8× too narrow — it understates uncertainty exactly in the small-sample regime you care about.

In [ ]:
show_fig("player_ci/figures/example_player_bands.png")

**Takeaway.** The right uncertainty band comes from resampling a player's PAs, not from
BART. But the bootstrap band is centered on the *raw* number, so it over-credits hot small
samples. Fixing the **center** (regressing it for sample size) is the true-talent estimate → notebook **04**.